# 00e Anchor Excel Profile

Purpose:
- profile the real anchor Excel workbook,
- infer SKU/quantity/date columns safely,
- export reproducible profile artifacts for synthetic generation calibration.


In [1]:
from pathlib import Path
from datetime import datetime
import json
import pandas as pd

ROOT = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling')
REPORTS_DIR = ROOT / 'outputs' / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_EXCEL_PATHS = [
    Path('/Users/k.e.oshada/Desktop/Mavin sir resources WMS hemas/RM ROP and Pallet requirement - SEP.xlsx'),
    Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/Forecast model train data optiwms/RM ROP and Pallet requirement  4- SEP.xlsx'),
]

def resolve_excel_path(candidates):
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f'No anchor Excel file found. Checked: {candidates}')

ANCHOR_EXCEL = resolve_excel_path(CANDIDATE_EXCEL_PATHS)
ANCHOR_EXCEL


PosixPath('/Users/k.e.oshada/Desktop/Mavin sir resources WMS hemas/RM ROP and Pallet requirement - SEP.xlsx')

In [2]:
xlsx = pd.ExcelFile(ANCHOR_EXCEL)
sheet_names = xlsx.sheet_names
sheet_names


['Active stock', 'Non moving', 'RM not stored in pallets', 'Sheet1']

In [3]:
def pick_anchor_sheet(names):
    ranked = [
        'active stock',
        'active_stock',
        'active',
    ]
    lower_map = {n.lower().strip(): n for n in names}
    for key in ranked:
        for n_low, n_orig in lower_map.items():
            if key in n_low:
                return n_orig
    return names[0]

anchor_sheet = pick_anchor_sheet(sheet_names)
anchor_sheet


'Active stock'

In [4]:
df = pd.read_excel(ANCHOR_EXCEL, sheet_name=anchor_sheet)
df.columns = [str(c).strip() for c in df.columns]
df.head(10)


,Material Code,Unnamed: 1,Description,Supply Plan,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Buffer days,future average,...,Pallet requirement.1,Unnamed: 28,Unnamed: 29,Issuing,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36
0,NaN,NaN,NaN,Jul SP,Aug SP,Sep SP,Oct SP,Nov SP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100036.0,Bags,CAUSTIC SODA,116865.440888,65374.215417,83043.700355,77736.826571,69301.968043,30.0,88715.094385,...,NaN,140.23,7.307352e+06,488.0,NaN,NaN,NaN,NaN,NaN,NaN
2,101054.0,Bags,CALCIUM CARBONATE ( GROUND ),79297.528709,73708.020695,49596.085217,57644.718956,61055.450093,30.0,63018.044085,...,73.0,37.53,2.708606e+06,227.0,NaN,NaN,NaN,NaN,NaN,NaN
3,100098.0,Drum,SORBITOL,79524.411769,65533.631292,51795.15625,61492.697155,58203.758094,30.0,66273.170563,...,55.0,120.14,7.086615e+06,2158.0,NaN,NaN,Total Racking,1290.0,NaN,1273+384
4,101293.0,Reel,FLUFF UNTREATED - GOLDEN ISLES G4881,63658.481755,24717.807518,54640.870026,45125.555026,53973.105026,30.0,56599.575899,...,59.0,139.04,1.063058e+07,96.0,NaN,NaN,NaN,NaN,NaN,NaN
5,100108.0,Bags,TALCUM POWDER,48838.939447,17584.315024,32702.066652,38951.115723,24893.581463,30.0,31648.350934,...,41.0,45.50,1.847896e+06,276.0,NaN,NaN,NaN,NaN,NaN,NaN
6,100323.0,Drum,BC COLOGNE BULK - IMPORTED,21973.045086,19397.41125,19546.901063,21821.528021,18504.07195,30.0,21519.345047,...,34.0,319.54,8.442746e+06,144.0,NaN,NaN,1st Layer,297.0,NaN,NaN
7,100094.0,Bags,SILICA 165 THICKENING,10152.968491,9819.050012,6611.939912,7833.125097,8212.370223,30.0,8590.555172,...,35.0,237.89,3.253398e+06,1321.0,NaN,NaN,NaN,NaN,NaN,NaN
8,101466.0,Reel,3D BUBBLE NON-WOVEN TOP SHEET 38GSM-90MM,4050.332761,1646.691275,3603.327376,2791.286176,3584.890096,30.0,3841.919851,...,45.0,1026.84,6.895911e+06,45.0,NaN,NaN,NaN,NaN,NaN,NaN
9,101580.0,Drum,SODIUM SILICATE,35518.185734,14181.362699,21850.583756,23693.252072,19548.081714,30.0,22958.293195,...,0.0,33.00,NaN,57.0,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
def choose_col(columns, candidates):
    low = {c.lower(): c for c in columns}
    for cand in candidates:
        for lc, orig in low.items():
            if cand in lc:
                return orig
    return None

sku_col = choose_col(df.columns, ['sku', 'item code', 'material code', 'material', 'code'])
qty_col = choose_col(df.columns, ['qty', 'quantity', 'rop', 'requirement', 'demand'])
date_col = choose_col(df.columns, ['date', 'month', 'period'])

sku_col, qty_col, date_col


('Material Code', 'Stacking quantity', 'lead time months')

In [6]:
profile = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'excel_path': str(ANCHOR_EXCEL),
    'anchor_sheet': anchor_sheet,
    'row_count': int(len(df)),
    'column_count': int(len(df.columns)),
    'columns': [str(c) for c in df.columns],
    'inferred_columns': {
        'sku_col': sku_col,
        'qty_col': qty_col,
        'date_col': date_col,
    },
}

if qty_col is not None:
    qty = pd.to_numeric(df[qty_col], errors='coerce')
    profile['qty_summary'] = {
        'non_null_count': int(qty.notna().sum()),
        'min': float(qty.min()) if qty.notna().any() else None,
        'p25': float(qty.quantile(0.25)) if qty.notna().any() else None,
        'median': float(qty.median()) if qty.notna().any() else None,
        'p75': float(qty.quantile(0.75)) if qty.notna().any() else None,
        'max': float(qty.max()) if qty.notna().any() else None,
    }

if sku_col is not None:
    sku_series = df[sku_col].astype(str).str.strip()
    profile['sku_summary'] = {
        'distinct_skus': int(sku_series.nunique(dropna=True)),
        'top_10_skus': sku_series.value_counts(dropna=True).head(10).to_dict(),
    }

profile


{'timestamp_utc': '2026-04-20T05:21:30.410140Z',
 'excel_path': '/Users/k.e.oshada/Desktop/Mavin sir resources WMS hemas/RM ROP and Pallet requirement - SEP.xlsx',
 'anchor_sheet': 'Active stock',
 'row_count': 313,
 'column_count': 37,
 'columns': ['Material Code',
  'Unnamed: 1',
  'Description',
  'Supply Plan',
  'Unnamed: 4',
  'Unnamed: 5',
  'Unnamed: 6',
  'Unnamed: 7',
  'Buffer days',
  'future average',
  'lead time',
  'lead time months',
  'EX',
  'variance (demand)',
  'Variance lead time demand',
  'ROP',
  'ROP in days',
  'Buffer stock',
  'Unnamed: 18',
  'Maximum stock',
  'Stacking quantity',
  'MOQ',
  'Difference',
  'Order Delivery',
  'Order Quantity',
  'Unnamed: 25',
  'Pallet requirement',
  'Pallet requirement.1',
  'Unnamed: 28',
  'Unnamed: 29',
  'Issuing',
  'Unnamed: 31',
  'Unnamed: 32',
  'Unnamed: 33',
  'Unnamed: 34',
  'Unnamed: 35',
  'Unnamed: 36'],
 'inferred_columns': {'sku_col': 'Material Code',
  'qty_col': 'Stacking quantity',
  'date_col': 

In [7]:
stamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
json_out = REPORTS_DIR / f'anchor_profile_metadata_{stamp}.json'
csv_out = REPORTS_DIR / f'anchor_profile_summary_{stamp}.csv'
latest_json = REPORTS_DIR / 'anchor_profile_metadata.json'
latest_csv = REPORTS_DIR / 'anchor_profile_summary.csv'

json_out.write_text(json.dumps(profile, indent=2), encoding='utf-8')
latest_json.write_text(json.dumps(profile, indent=2), encoding='utf-8')

table = pd.DataFrame([
    {
        'excel_path': profile['excel_path'],
        'anchor_sheet': profile['anchor_sheet'],
        'row_count': profile['row_count'],
        'column_count': profile['column_count'],
        'sku_col': profile['inferred_columns']['sku_col'],
        'qty_col': profile['inferred_columns']['qty_col'],
        'date_col': profile['inferred_columns']['date_col'],
        'distinct_skus': profile.get('sku_summary', {}).get('distinct_skus'),
        'qty_min': profile.get('qty_summary', {}).get('min'),
        'qty_median': profile.get('qty_summary', {}).get('median'),
        'qty_max': profile.get('qty_summary', {}).get('max'),
        'timestamp_utc': profile['timestamp_utc'],
    }
])
table.to_csv(csv_out, index=False)
table.to_csv(latest_csv, index=False)

print('WROTE', json_out)
print('WROTE', csv_out)
print('WROTE', latest_json)
print('WROTE', latest_csv)
